# define prompt

In [1]:
SYSTEM_PROMPT = """
You are a Greek–Latvian Bible word alignment engine.

INPUT FORMAT:
First line(s): Latvian verse.
After newline: Greek verse.
Each Latvian line corresponds to same Greek line.

TASK:
- Preserve Greek word order exactly.
- For each Greek token, map zero or more Latvian words.
- Output ONLY valid JSON.
- No explanations.
- No markdown.
- No comments.

OUTPUT FORMAT:

{
  "verse": "...",
  "greek_words": [
    {
      "index": integer,
      "greek": "string",
      "latvian": ["string", "..."]
    }
  ],
  "leftover_latvian": ["string", "..."]
}

If Latvian words remain unmapped, include them in leftover_latvian.
If none remain, return empty list.
"""

In [3]:
from openai import OpenAI
import json

client = OpenAI(
    base_url="http://localhost:11435/v1",
    api_key="ollama"
)

def get_alignment(latvian_text, greek_text):
    user_input = latvian_text.strip() + "\n" + greek_text.strip()

    response = client.chat.completions.create(
        model="qwen2.5",
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ],
    )

    return json.loads(response.choices[0].message.content)

In [11]:
import pandas as pd

strongs_df = pd.read_csv("strongs.csv")
morph_df = pd.read_csv("morphology.csv")
lv65_df = pd.read_csv("latvian_full65.csv")

/tmp/ipykernel_1953268/2708557476.py:4: DtypeWarning: Columns (0: information-status, 1: presentation-before, 2: contrast-group) have mixed types. Specify dtype option on import or set low_memory=False.
  morph_df = pd.read_csv("morphology.csv")


In [13]:
import unicodedata
def raccs_common(text):
    d = {
        #ord('\N{COMBINING ACUTE ACCENT}'):None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('’'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        }
    return unicodedata.normalize('NFC', text).translate(d)
print(strongs_df["form"].apply(raccs_common))
print(morph_df["form"].apply(raccs_common))

0            Παῦλος
1            κλητὸς
2         ἀπόστολος
3           Χριστοῦ
4             Ἰησοῦ
            ...    
138988        χάρις
138989         μετὰ
138990       πάντων
138991         ὑμῶν
138992         Ἀμήν
Name: form, Length: 138993, dtype: str
0           Βίβλος
1         γενέσεως
2            Ἰησοῦ
3          Χριστοῦ
4             υἱοῦ
            ...   
132351         τοῦ
132352      κυρίου
132353       Ἰησοῦ
132354        μετὰ
132355      πάντων
Name: form, Length: 132356, dtype: str


In [42]:
def get_verse_wordsorted(df, book, chapter, verse):
    return " ".join(row["form"] for _, row in df.query(
        "book == @book and chapter == @chapter and verse == @verse"
    ).sort_values("word").iterrows())

def get_verse(df, book, chapter, verse):
    return " ".join(row["text"] for  _, row in df.query(
        "book == @book and chapter == @chapter and verse == @verse"
    ).iterrows())

def get_chapter(df, book, chapter):
    ch = df.query(
        "book == @book and chapter == @chapter"
    )
    max_verse = ch["verse"].max()
    print(df)

get_chapter(lv65_df, "1_corinthians",  1)
#print(get_verse(strongs_df, "1_corinthians",  1 ,1).head())
test_verse=get_verse_wordsorted(strongs_df, "1_corinthians",  1 ,1)
test_verse65=get_verse(lv65_df, "1_corinthians",  1 ,1)
#print([test_verse.iloc[i] for i in range(test_verse['word'].max())])
#print([str(row["word"])+" "+row["form"] for _, row in test_verse.iterrows()])
#print(" ".join(row["form"] for _, row in test_verse.iterrows()))
#print(" ".join(row["text"] for  _, row in test_verse65.iterrows()))
print(test_verse)
print(test_verse65)

            book  chapter  verse  \
0        matthew        1      1   
1        matthew        1      2   
2        matthew        1      3   
3        matthew        1      4   
4        matthew        1      5   
...          ...      ...    ...   
7951  revelation       22     17   
7952  revelation       22     18   
7953  revelation       22     19   
7954  revelation       22     20   
7955  revelation       22     21   

                                                   text  
0     Jēzus Kristus Dāvida dēla Ābrahāma dēla cilts ...  
1     Ābrahāms dzemdināja Īzāku Īzāks dzemdināja Jēk...  
2     Un Jūda dzemdināja no Tamāras Ferecu un Caru F...  
3     un Arāms dzemdināja Aminadabu Aminadabs dzemdi...  
4     Salmons dzemdināja no Rahabas Boāsu Boāss dzem...  
...                                                 ...  
7951  Un Gars un līgava saka Nāc Un kas to dzird lai...  
7952  Es apliecinu katram kas dzird šās grāmatas pra...  
7953  Ja kas ko atņem no šīs grāmatas praviet

In [52]:
import pandas as pd

def prepare_grouped(strongs_df, morph_df, lv_df):
    strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
    morph_g = morph_df.groupby(["book", "chapter", "verse"])
    lv_g = lv_df.groupby(["book", "chapter", "verse"])
    return strongs_g, morph_g, lv_g

def merge_greek_word_data(strongs_verse, morph_verse):
    strongs_verse = strongs_verse.sort_values("word")
    morph_verse = morph_verse.sort_values("word")

    merged = pd.merge(
        strongs_verse,
        morph_verse,
        on=["book", "chapter", "verse", "word"],
        how="left",      # keep all Greek tokens
        suffixes=("_strong", "_morph")
    )

    return merged

def build_morph_lookup(morph_verse):
    morph_verse = morph_verse.sort_values("word")

    lemma_map = {}

    for _, row in morph_verse.iterrows():
        lemma = row["lemma"]

        if lemma not in lemma_map:
            lemma_map[lemma] = []

        lemma_map[lemma].append(row.to_dict())

    return lemma_map

def enrich_strongs_with_morph(strongs_verse, morph_verse):

    strongs_verse = strongs_verse.sort_values("word")
    morph_lookup = build_morph_lookup(morph_verse)

    enriched_tokens = []

    for _, s_row in strongs_verse.iterrows():
        lemma = s_row["lemma"]

        morph_data = {}

        if lemma in morph_lookup and morph_lookup[lemma]:
            morph_data = morph_lookup[lemma].pop(0)  # consume in order

        enriched_tokens.append({
            "index": int(s_row["word"]),
            "form": s_row["form"],
            "lemma": lemma,
            "strong_num": s_row.get("strong_num", ""),
            "translit_title": s_row.get("translit_title", ""),
            "morphology": morph_data.get("morphology", ""),
            "pos": morph_data.get("part-of-speech", "")
        })

    return enriched_tokens

def validate_word_indices(df):
    indices = df["word"].tolist()
    expected = list(range(max(indices) + 1))

    if indices != expected:
        raise ValueError(f"Word index mismatch! Expected {expected}, got {indices}")

def build_verse_object(book, chapter, verse, strongs_g, morph_g, lv_g):

    key = (book, chapter, verse)

    if key not in strongs_g.groups:
        return None

    if key not in lv_g.groups:
        return None

    strongs_verse = strongs_g.get_group(key)
    morph_verse = morph_g.get_group(key) if key in morph_g.groups else pd.DataFrame()

    enriched_tokens = enrich_strongs_with_morph(strongs_verse, morph_verse)

    latvian_text = lv_g.get_group(key).iloc[0]["text"]

    greek_text = " ".join([t["form"] for t in enriched_tokens])

    return {
        "book": book,
        "chapter": chapter,
        "verse": verse,
        "greek_text": greek_text,
        "latvian_text": latvian_text,
        "greek_word_data": enriched_tokens
    }

    
def process_book_full(book, strongs_df, morph_df, lv_df):

    strongs_g, morph_g, lv_g = prepare_grouped(strongs_df, morph_df, lv_df)

    results = []

    # Drive iteration from Latvian (verse-level dataset)
    for key in sorted(lv_g.groups.keys()):
        b, chapter, verse = key

        if b != book:
            continue

        verse_obj = build_verse_object(
            b, chapter, verse,
            strongs_g, morph_g, lv_g
        )

        if verse_obj:
            results.append(verse_obj)

    return results

book_data = process_book_full(
    "1_corinthians",
    strongs_df,
    morph_df,
    lv65_df
)

print(book_data[0])

KeyError: 'lemma'